# ETL Process: Data Warehouse Implementation

This notebook implements the complete ETL process to transform cleaned source data into a star schema data warehouse.

## Process Overview
1. **Extract**: Load cleaned source data
2. **Transform**: Create dimensional and fact tables
3. **Validate**: Quality checks and referential integrity
4. **Load**: Export to CSV and generate reports

## Setup: Imports and Configuration

In [24]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from pathlib import Path
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('etl_execution.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuration
SOURCE_DIR = "."
OUTPUT_DIR = "./dw_output"
DIMS_DIR = os.path.join(OUTPUT_DIR, "dimensions")
FACTS_DIR = os.path.join(OUTPUT_DIR, "facts")
LOGS_DIR = os.path.join(OUTPUT_DIR, "logs")

# Create output directories
for dir_path in [DIMS_DIR, FACTS_DIR, LOGS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

logger.info("ETL Process Started")
logger.info(f"Output directories created: {OUTPUT_DIR}")

2026-04-04 15:24:29,150 - INFO - ETL Process Started
2026-04-04 15:24:29,153 - INFO - Output directories created: ./dw_output


## Phase 1: Extraction

Load all cleaned source datasets

In [25]:
# Load raw source data
logger.info("=" * 60)
logger.info("PHASE 1: EXTRACTION & CLEANING")
logger.info("=" * 60)

try:
    # Load raw files
    df_pop_raw = pd.read_csv("ine_population_data.csv", encoding="latin1", sep=";", skiprows=3)
    logger.info(f"✓ Population data extracted: {len(df_pop_raw)} rows")
    
    df_substations_raw = pd.read_csv("postos-transformacao-distribuicao.csv", sep=";")
    logger.info(f"✓ Substations data extracted: {len(df_substations_raw)} rows")
    
    df_weather_sp_raw = pd.read_csv("porto_serra_do_pilar_weather.csv")
    logger.info(f"✓ Weather (Serra do Pilar) extracted: {len(df_weather_sp_raw)} rows")
    
    df_weather_pr_raw = pd.read_csv("porto_pedras_rubras_weather.csv")
    logger.info(f"✓ Weather (Pedras Rubras) extracted: {len(df_weather_pr_raw)} rows")
    
    df_consumption_pt1_raw = pd.read_csv("consumo_horario_mobilidade_eletrica_01_a_10.csv", sep=";")
    df_consumption_pt2_raw = pd.read_csv("consumo_horario_mobilidade_eletrica_11_a_18.csv", sep=";")
    logger.info(f"✓ Consumption data extracted: {len(df_consumption_pt1_raw) + len(df_consumption_pt2_raw)} rows")
    
    # Raw charging data will be loaded with proper encoding
    df_charging_raw = pd.read_csv(
        "ine_charging_points_number_location.csv", 
        encoding='iso-8859-1',
        sep=';',
        on_bad_lines='skip',
        engine='python'
    )
    logger.info(f"✓ Raw Charging data loaded: {len(df_charging_raw)} rows")
    
    logger.info("Extraction completed successfully")
except FileNotFoundError as e:
    logger.error(f"File not found: {e}")
    raise
except Exception as e:
    logger.error(f"Error during extraction: {e}")
    raise

2026-04-04 15:24:32,335 - INFO - ============================================================
2026-04-04 15:24:32,336 - INFO - PHASE 1: EXTRACTION & CLEANING
2026-04-04 15:24:32,339 - INFO - ============================================================
2026-04-04 15:24:32,360 - INFO - ✓ Population data extracted: 180 rows
2026-04-04 15:24:32,774 - INFO - ✓ Substations data extracted: 72349 rows
2026-04-04 15:24:32,780 - INFO - ✓ Weather (Serra do Pilar) extracted: 1520 rows
2026-04-04 15:24:32,787 - INFO - ✓ Weather (Pedras Rubras) extracted: 1520 rows
2026-04-04 15:24:51,422 - INFO - ✓ Consumption data extracted: 9315214 rows
2026-04-04 15:24:51,430 - INFO - ✓ Raw Charging data loaded: 7 rows
2026-04-04 15:24:51,431 - INFO - Extraction completed successfully


## 1.1: Clean Population Data

In [26]:
# Clean population data
df_pop = df_pop_raw.copy()
df_pop.drop(0, inplace=True)

# Rename columns
df_pop.columns = [
    "year", "region", "growth_rate", "density", 
    "cities", "freguesias", "vilas", "extra"
]

# Fill year forward (for merged cells in original data)
df_pop["year"] = df_pop["year"].ffill()

# Select data of interest (first 73 rows contain Porto Metropolitan Area data)
df_pop = df_pop.iloc[0:72]

# Split region into code and name
df_pop[["region_code", "region_name"]] = df_pop["region"].str.split(":", n=1, expand=True)
df_pop = df_pop.drop('region', axis=1)

# Remove extra column and clean region names
df_pop = df_pop.drop('extra', axis=1)
df_pop["region_name"] = df_pop["region_name"].str.strip()

# Get unique regions for filtering other datasets
unique_regions = df_pop["region_name"].dropna().unique().tolist()

logger.info(f"✓ Population data cleaned: {len(df_pop)} rows")
logger.info(f"  Columns: {df_pop.columns.tolist()}")
logger.info(f"  Regions: {len(unique_regions)} municipalities")

2026-04-04 15:24:51,464 - INFO - ✓ Population data cleaned: 72 rows
2026-04-04 15:24:51,465 - INFO -   Columns: ['year', 'growth_rate', 'density', 'cities', 'freguesias', 'vilas', 'region_code', 'region_name']
2026-04-04 15:24:51,467 - INFO -   Regions: 18 municipalities


## 1.2: Clean Substations Data

In [27]:
# Clean substations data - filter for Porto Metropolitan Area
df_substations = df_substations_raw[
    df_substations_raw["Municipality"].isin(unique_regions)
]

logger.info(f"✓ Substations data cleaned: {len(df_substations)} rows")
logger.info(f"  Filtered to Porto Metropolitan Area")

2026-04-04 15:24:54,715 - INFO - ✓ Substations data cleaned: 8412 rows
2026-04-04 15:24:54,718 - INFO -   Filtered to Porto Metropolitan Area


## 1.3: Clean Weather Data

In [28]:
# Clean weather data for both stations
for df_raw, station_name in [(df_weather_sp_raw, "Serra do Pilar"), 
                              (df_weather_pr_raw, "Pedras Rubras")]:
    # Convert date column
    df_raw["date"] = pd.to_datetime(df_raw["date"])
    
    # Extract datetime components
    df_raw["year"] = df_raw["date"].dt.year
    df_raw["month"] = df_raw["date"].dt.month
    df_raw["day"] = df_raw["date"].dt.day
    df_raw["time"] = df_raw["date"].dt.time
    
    # Drop columns with all null values
    df_raw.drop(columns=["snow", "wdir", "tsun"], inplace=True)
    
    logger.info(f"  Weather ({station_name}) cleaned: {len(df_raw)} rows")

# Assign cleaned data
df_weather_sp = df_weather_sp_raw
df_weather_pr = df_weather_pr_raw

logger.info(f"✓ Weather data cleaned for both stations")

2026-04-04 15:24:57,235 - INFO -   Weather (Serra do Pilar) cleaned: 1520 rows
2026-04-04 15:24:57,247 - INFO -   Weather (Pedras Rubras) cleaned: 1520 rows
2026-04-04 15:24:57,249 - INFO - ✓ Weather data cleaned for both stations


## 1.4: Clean Consumption Data

In [29]:
# Clean consumption data - combine and filter for Porto Metropolitan Area
df_consumption_pt1 = df_consumption_pt1_raw[
    df_consumption_pt1_raw["Municipality"].isin(unique_regions)
]
df_consumption_pt2 = df_consumption_pt2_raw[
    df_consumption_pt2_raw["Municipality"].isin(unique_regions)
]

# Combine both parts
df_consumption = pd.concat(
    [df_consumption_pt1, df_consumption_pt2], 
    axis=0, 
    ignore_index=True
)

logger.info(f"✓ Consumption data cleaned: {len(df_consumption)} rows")
logger.info(f"  Combined from 2 datasets and filtered to Porto Metropolitan Area")

2026-04-04 15:25:01,316 - INFO - ✓ Consumption data cleaned: 1027232 rows
2026-04-04 15:25:01,317 - INFO -   Combined from 2 datasets and filtered to Porto Metropolitan Area


## 1.5: Clean Charging Data

In [30]:
def clean_charging_data(filename):
    """
    Clean the raw INE charging points data.
    
    File structure:
    - Line 8: Months (Dezembro de 2024 in col 1, empty cols 2-4, Novembro de 2024 in col 6, etc.)
    - Line 10: Socket types (Total, Mennekes, CHAdeMO, CCS, Industrial repeating)
    - Line 12+: Data rows (location in col 0, values in cols 1+)
    - Each month takes 5 columns for the 5 socket types
    """
    
    with open(filename, 'r', encoding='iso-8859-1') as f:
        lines = f.readlines()
    
    # Extract header information
    months_row = [x.strip() for x in lines[8].split(';')]
    socket_types_row = [x.strip() for x in lines[10].split(';')]
    
    # Build month-year mapping for each column position
    col_to_month = {}
    month_map = {
        'Janeiro': 1, 'Fevereiro': 2, 'Março': 3, 'Abril': 4,
        'Maio': 5, 'Junho': 6, 'Julho': 7, 'Agosto': 8,
        'Setembro': 9, 'Outubro': 10, 'Novembro': 11, 'Dezembro': 12
    }
    
    current_month_year = None
    for col_idx, val in enumerate(months_row):
        if val and 'de 20' in val:
            parts = val.split()
            month_name = parts[0]
            year = int(parts[-1])
            month_num = month_map.get(month_name, 1)
            current_month_year = (year, month_num)
        
        if current_month_year:
            col_to_month[col_idx] = current_month_year
    
    # Process data rows
    data_rows = []
    for line_idx in range(12, len(lines)):
        line = lines[line_idx].strip()
        if not line:
            continue
        
        cols = [x.strip() for x in line.split(';')]
        location = cols[0]
        
        if not location:
            continue
        
        # Remove location code
        if ':' in location:
            location = location.split(':', 1)[1].strip()
        
        # Process each value column
        for col_idx, val in enumerate(cols[1:], start=1):
            if col_idx not in col_to_month or col_idx >= len(socket_types_row):
                continue
            
            socket_type = socket_types_row[col_idx]
            year, month = col_to_month[col_idx]
            
            # Parse value
            if '- -' in val or val == '' or val == 'nan':
                count = None
            else:
                try:
                    count = int(val)
                except (ValueError, TypeError):
                    count = None
            
            data_rows.append({
                'location': location,
                'year': year,
                'month': month,
                'outlet_type': socket_type,
                'charging_points': count
            })
    
    df_cleaned = pd.DataFrame(data_rows)
    
    if len(df_cleaned) > 0:
        df_cleaned = df_cleaned.drop_duplicates()
        df_cleaned = df_cleaned.sort_values(['year', 'month', 'location'], ignore_index=True)
    
    return df_cleaned

# Clean the raw charging data
df_charging = clean_charging_data("ine_charging_points_number_location.csv")

logger.info(f"✓ Charging data cleaned: {len(df_charging)} rows")
if len(df_charging) > 0:
    logger.info(f"  Date range: {df_charging['year'].min()}-{df_charging['month'].min():02d} to {df_charging['year'].max()}-{df_charging['month'].max():02d}")
    logger.info(f"  Locations: {df_charging['location'].nunique()}")
    logger.info(f"  Outlet types: {len(df_charging['outlet_type'].unique())}")

2026-04-04 15:25:03,466 - INFO - ✓ Charging data cleaned: 4355 rows
2026-04-04 15:25:03,471 - INFO -   Date range: 2021-01 to 2024-12
2026-04-04 15:25:03,476 - INFO -   Locations: 35
2026-04-04 15:25:03,478 - INFO -   Outlet types: 6


## Phase 2: Transformation

### 2.1 Create DIM_LOCATION

In [31]:
logger.info("=" * 60)
logger.info("PHASE 2: TRANSFORMATION")
logger.info("=" * 60)

# Extract unique municipalities from population data
dim_location = df_pop[['region_code', 'region_name']].drop_duplicates().reset_index(drop=True)

# Rename columns to match dimension schema
dim_location.columns = ['municipality_code', 'municipality_name']

# Assign location_id (surrogate key)
dim_location['location_id'] = range(1, len(dim_location) + 1)

# Add district information (derived from municipality name or mapping)
# For simplicity, we'll use municipality as both municipality and district
# In real scenario, you'd use a mapping table
dim_location['district_code'] = dim_location['municipality_code']
dim_location['district_name'] = dim_location['municipality_name']
dim_location['parish_code'] = None  # Optional data not available in cleaned sources
dim_location['parish_name'] = None

# Reorder columns according to schema
dim_location = dim_location[[
    'location_id',
    'municipality_code',
    'municipality_name',
    'parish_code',
    'parish_name',
    'district_code',
    'district_name'
]]

logger.info(f"✓ DIM_LOCATION created: {len(dim_location)} rows")
display(dim_location.head(10))

2026-04-04 15:25:08,398 - INFO - ============================================================
2026-04-04 15:25:08,400 - INFO - PHASE 2: TRANSFORMATION
2026-04-04 15:25:08,402 - INFO - ============================================================
2026-04-04 15:25:08,414 - INFO - ✓ DIM_LOCATION created: 18 rows


,location_id,municipality_code,municipality_name,parish_code,parish_name,district_code,district_name
0,1,11A,Área Metropolitana do Porto,None,None,11A,Área Metropolitana do Porto
1,2,11A0104,Arouca,None,None,11A0104,Arouca
2,3,11A0107,Espinho,None,None,11A0107,Espinho
3,4,11A1304,Gondomar,None,None,11A1304,Gondomar
4,5,11A1306,Maia,None,None,11A1306,Maia
5,6,11A1308,Matosinhos,None,None,11A1308,Matosinhos
6,7,11A0113,Oliveira de Azeméis,None,None,11A0113,Oliveira de Azeméis
7,8,11A1310,Paredes,None,None,11A1310,Paredes
8,9,11A1312,Porto,None,None,11A1312,Porto
9,10,11A1313,Póvoa de Varzim,None,None,11A1313,Póvoa de Varzim


### 2.2 Create DIM_TIME

In [32]:
# Extract min and max dates from all datasets
dates = []

# From population: year
pop_dates = pd.to_datetime(df_pop['year'].astype(str))
dates.extend(pop_dates.tolist())

# From weather
if 'date' in df_weather_sp.columns:
    dates.extend(pd.to_datetime(df_weather_sp['date']).tolist())
if 'date' in df_weather_pr.columns:
    dates.extend(pd.to_datetime(df_weather_pr['date']).tolist())

# From consumption
if 'Period' in df_consumption.columns:
    dates.extend(pd.to_datetime(df_consumption['Period'], errors='coerce').dropna().tolist())

# From charging
charging_dates = pd.to_datetime(
    df_charging['year'].astype(str) + '-' + 
    df_charging['month'].astype(str) + '-01',
    errors='coerce'
)
dates.extend(charging_dates.tolist())

# Remove None and duplicates
dates = pd.to_datetime([d for d in dates if d is not None]).unique()
dates = sorted(dates)

min_date = pd.Timestamp(dates[0])
max_date = pd.Timestamp(dates[-1])

logger.info(f"Date range: {min_date.date()} to {max_date.date()}")

# Generate complete datetime range
date_range = pd.date_range(start=min_date, end=max_date, freq='H')

dim_time = pd.DataFrame({
    'datetime': date_range
})

# Extract datetime components
dim_time['date'] = dim_time['datetime'].dt.date
dim_time['year'] = dim_time['datetime'].dt.year
dim_time['month'] = dim_time['datetime'].dt.month
dim_time['day'] = dim_time['datetime'].dt.day
dim_time['hour'] = dim_time['datetime'].dt.hour

# Create level keys (surrogate keys for each level)
# Year level key
dim_time['time_id'] = dim_time['year'] - dim_time['year'].min() + 1

# Month level key
dim_time['month_id'] = (
    (dim_time['year'] - dim_time['year'].min()) * 12 + dim_time['month']
)

# Day level key
dim_time['day_id'] = (
    dim_time['datetime'].astype('int64') // (10**9 * 86400)
)

# Hour level key
dim_time['hour_id'] = range(len(dim_time))

# Reorder columns
dim_time = dim_time[[
    'hour_id',
    'day_id',
    'month_id',
    'time_id',
    'hour',
    'day',
    'month',
    'year',
    'date',
    'datetime'
]]

logger.info(f"✓ DIM_TIME created: {len(dim_time)} rows")
logger.info(f"  Time range: {dim_time['datetime'].min()} to {dim_time['datetime'].max()}")
display(dim_time.head(10))

2026-04-04 15:25:18,280 - INFO - Date range: 2021-01-01 to 2026-02-28
C:\Users\iscap\AppData\Local\Temp\ipykernel_9236\3726882397.py:36: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  date_range = pd.date_range(start=min_date, end=max_date, freq='H')
2026-04-04 15:25:18,360 - INFO - ✓ DIM_TIME created: 45217 rows
2026-04-04 15:25:18,366 - INFO -   Time range: 2021-01-01 00:00:00 to 2026-02-28 00:00:00


,hour_id,day_id,month_id,time_id,hour,day,month,year,date,datetime
0,0,18628,1,1,0,1,1,2021,2021-01-01,2021-01-01 00:00:00
1,1,18628,1,1,1,1,1,2021,2021-01-01,2021-01-01 01:00:00
2,2,18628,1,1,2,1,1,2021,2021-01-01,2021-01-01 02:00:00
3,3,18628,1,1,3,1,1,2021,2021-01-01,2021-01-01 03:00:00
4,4,18628,1,1,4,1,1,2021,2021-01-01,2021-01-01 04:00:00
5,5,18628,1,1,5,1,1,2021,2021-01-01,2021-01-01 05:00:00
6,6,18628,1,1,6,1,1,2021,2021-01-01,2021-01-01 06:00:00
7,7,18628,1,1,7,1,1,2021,2021-01-01,2021-01-01 07:00:00
8,8,18628,1,1,8,1,1,2021,2021-01-01,2021-01-01 08:00:00
9,9,18628,1,1,9,1,1,2021,2021-01-01,2021-01-01 09:00:00


### 2.3 Create DIM_WEATHER

In [33]:
# Combine weather data from both stations
df_weather_sp['date'] = pd.to_datetime(df_weather_sp['date'])
df_weather_pr['date'] = pd.to_datetime(df_weather_pr['date'])

# Merge both weather sources (average their values)
weather_merged = pd.merge(
    df_weather_sp[['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']],
    df_weather_pr[['date', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres']],
    on='date',
    how='outer',
    suffixes=('_sp', '_pr')
)

# Calculate averages
dim_weather = pd.DataFrame({
    'date': weather_merged['date'],
    'avg_temperature': weather_merged[['tavg_sp', 'tavg_pr']].mean(axis=1),
    'min_temperature': weather_merged[['tmin_sp', 'tmin_pr']].min(axis=1),
    'max_temperature': weather_merged[['tmax_sp', 'tmax_pr']].max(axis=1),
    'precipitation': weather_merged[['prcp_sp', 'prcp_pr']].sum(axis=1),
    'wind_speed': weather_merged[['wspd_sp', 'wspd_pr']].mean(axis=1),
    'pressure': weather_merged[['pres_sp', 'pres_pr']].mean(axis=1)
})

# Link to time dimension (day level)
# Convert dim_time date to datetime64[ns] to match dim_weather for merge
dim_time_for_merge = dim_time[['date', 'day_id', 'month_id', 'time_id']].drop_duplicates().copy()
dim_time_for_merge['date'] = pd.to_datetime(dim_time_for_merge['date'])

dim_weather = dim_weather.merge(
    dim_time_for_merge,
    on='date',
    how='left'
)

# Assign weather_id
dim_weather['weather_id'] = range(1, len(dim_weather) + 1)
dim_weather.rename(columns={'time_id': 'time_id_ref'}, inplace=True)
dim_weather['time_id'] = dim_weather['day_id']  # Reference day level key

# Reorder columns
dim_weather = dim_weather[[
    'weather_id',
    'time_id',
    'avg_temperature',
    'min_temperature',
    'max_temperature',
    'precipitation',
    'wind_speed',
    'pressure'
]]

logger.info(f"✓ DIM_WEATHER created: {len(dim_weather)} rows")
display(dim_weather.head(10))

2026-04-04 15:25:26,816 - INFO - ✓ DIM_WEATHER created: 1520 rows


,weather_id,time_id,avg_temperature,min_temperature,max_temperature,precipitation,wind_speed,pressure
0,1,18993,17.20,11.6,23.7,0.0,16.90,1024.55
1,2,18994,13.85,10.6,23.7,4.3,8.45,1029.25
2,3,18995,11.95,8.8,17.2,0.3,14.35,1024.90
3,4,18996,12.00,8.5,15.3,9.7,18.35,1017.75
4,5,18997,9.75,7.1,13.4,10.4,10.35,1021.70
5,6,18998,8.20,4.1,13.3,0.3,6.40,1024.35
6,7,18999,10.05,6.4,14.1,0.3,13.05,1031.45
7,8,19000,7.25,2.3,14.1,0.3,6.45,1032.85
8,9,19001,12.35,6.8,13.7,17.5,8.15,1028.25
9,10,19002,12.30,8.9,13.6,2.3,10.25,1026.70


### 2.4 Create DIM_SUBSTATIONS

In [34]:
# Process substations data
dim_substations = df_substations.copy()

# Assign substation_id (surrogate key)
dim_substations['substation_id'] = range(1, len(dim_substations) + 1)

# Rename and map columns to schema
dim_substations = dim_substations.rename(columns={
    'Installation Code': 'installation_code',
    'Municipality': 'municipality_name'
})

# Map to location dimension to get municipality_code
dim_substations = dim_substations.merge(
    dim_location[['municipality_code', 'municipality_name', 'district_name']],
    on='municipality_name',
    how='left'
)

# Assign level keys (using month_id as proxy for data level)
dim_substations['municipality_code_lk'] = dim_substations['municipality_code']

# Select and reorder columns
columns_to_keep = [
    'substation_id',
    'installation_code',
    'municipality_code_lk',
    'municipality_name',
    'district_name'
]

# Add optional columns if they exist
optional_cols = [
    'Geographic Coordinates',  # Geographic coordinates
    'Installed Power [kVA]',
    'Contracted Power [kVA]',
    'Usage Level [%]',
    'Number of Clients',
    'Construction Type'
]

for col in optional_cols:
    if col in dim_substations.columns:
        columns_to_keep.append(col)

dim_substations = dim_substations[columns_to_keep]

# Rename to match schema
rename_map = {
    'municipality_code_lk': 'municipality_code',
    'Geographic Coordinates': 'geographic_coordinates',
    'Installed Power [kVA]': 'installed_power',
    'Contracted Power [kVA]': 'contracted_power',
    'Usage Level [%]': 'usage_level',
    'Number of Clients': 'number_of_clients',
    'Construction Type': 'construction_type'
}

dim_substations = dim_substations.rename(columns=rename_map)

logger.info(f"✓ DIM_SUBSTATIONS created: {len(dim_substations)} rows")
display(dim_substations.head(10))

2026-04-04 15:25:30,868 - INFO - ✓ DIM_SUBSTATIONS created: 8412 rows


,substation_id,installation_code,municipality_code,municipality_name,district_name,geographic_coordinates,installed_power,contracted_power,usage_level,number_of_clients,construction_type
0,1,1312D2103700,11A1312,Porto,Porto,"41.183677856847204, -8.608945507725844",630,1317.35,0%-19%,201,Cabine metálica (monobloco)
1,2,1316D2004400,11A1316,Vila do Conde,Vila do Conde,"41.336527228320506, -8.645671024999814",400,1610.01,20%-39%,216,Cabine alta
2,3,1316D2016800,11A1316,Vila do Conde,Vila do Conde,"41.27164654205575, -8.724611320666718",630,2047.95,20%-39%,336,Cabine baixa em edifício próprio
3,4,1316D2057000,11A1316,Vila do Conde,Vila do Conde,"41.279717670991126, -8.679046573660683",400,N/D,0%-19%,<20,Cabine pré-fabricada
4,5,1317D2070300,11A1317,Vila Nova de Gaia,Vila Nova de Gaia,"41.04344892828184, -8.542996018121055",250,640.00,20%-39%,71,Cabine baixa em edifício próprio
5,6,1317D2051700,11A1317,Vila Nova de Gaia,Vila Nova de Gaia,"41.11455052390216, -8.61403627288223",630,1178.46,0%-19%,188,Cabine baixa em edifício próprio
6,7,1317D2033500,11A1317,Vila Nova de Gaia,Vila Nova de Gaia,"41.13972799931655, -8.636187844159922",630,592.77,0%-19%,91,Cabine baixa em edifício próprio
7,8,1308D2023400,11A1308,Matosinhos,Matosinhos,"41.25329240728268, -8.721106859919177",400,2580.12,60%-79%,373,Cabine alta
8,9,1308D2071200,11A1308,Matosinhos,Matosinhos,"41.20898585173012, -8.687992786282576",1890,4095.66,20%-39%,185,Cabine baixa integrada em edifício
9,10,1308D2009700,11A1308,Matosinhos,Matosinhos,"41.196189239991355, -8.704738220997905",400,1140.09,20%-39%,150,Cabine alta


### 2.5 Create FACT_CONSUMPTION

In [35]:
# Process consumption data
fact_consumption = df_consumption.copy()

# Rename datetime column - handle mixed timezones
if 'Date/Time' in fact_consumption.columns:
    fact_consumption['datetime'] = pd.to_datetime(fact_consumption['Date/Time'], utc=True)
    fact_consumption['datetime'] = fact_consumption['datetime'].dt.tz_localize(None)
elif 'Period' in fact_consumption.columns:
    fact_consumption['datetime'] = pd.to_datetime(fact_consumption['Period'], utc=True)
elif 'Date' in fact_consumption.columns:
    fact_consumption['datetime'] = pd.to_datetime(fact_consumption['Date'])

# Extract hour information from the datetime
if 'datetime' in fact_consumption.columns and pd.api.types.is_datetime64_any_dtype(fact_consumption['datetime']):
    fact_consumption['date'] = fact_consumption['datetime'].dt.date
    fact_consumption['hour'] = fact_consumption['datetime'].dt.hour

# Map municipality to location_id
fact_consumption = fact_consumption.merge(
    dim_location[['municipality_name', 'location_id']],
    left_on='Municipality',
    right_on='municipality_name',
    how='left'
)

# Map datetime to time_id (hour level)
if 'datetime' in fact_consumption.columns and pd.api.types.is_datetime64_any_dtype(fact_consumption['datetime']):
    fact_consumption = fact_consumption.merge(
        dim_time[['datetime', 'hour_id', 'day_id']],
        on='datetime',
        how='left'
    )
    
    # Map to weather (using day_id)
    fact_consumption = fact_consumption.merge(
        dim_weather[['time_id', 'weather_id']],
        left_on='day_id',
        right_on='time_id',
        how='left'
    )

# Rename energy consumed column
if 'Energy Consumed' in fact_consumption.columns:
    fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['Energy Consumed'], errors='coerce')
elif 'Value' in fact_consumption.columns:
    fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['Value'], errors='coerce')
elif 'value' in fact_consumption.columns:
    fact_consumption['energy_consumed'] = pd.to_numeric(fact_consumption['value'], errors='coerce')

# Select fact table columns  
available_cols = ['location_id', 'hour_id', 'day_id', 'weather_id', 'energy_consumed']
selected_cols = [col for col in available_cols if col in fact_consumption.columns]
fact_consumption = fact_consumption[selected_cols]

# Remove rows with missing key references
fact_consumption = fact_consumption.dropna(subset=['location_id', 'energy_consumed'])

logger.info(f"✓ FACT_CONSUMPTION created: {len(fact_consumption)} rows")
display(fact_consumption.head(10))

2026-04-04 15:25:43,751 - INFO - ✓ FACT_CONSUMPTION created: 1027232 rows


,location_id,hour_id,day_id,weather_id,energy_consumed
0,11,41620,20362,1370,0.0
1,11,41711,20365,1373,0.0
2,11,42239,20387,1395,0.0
3,11,42123,20383,1391,0.0
4,11,41843,20371,1379,0.0
5,11,41716,20366,1374,0.0
6,11,41736,20367,1375,0.0
7,11,41906,20374,1382,0.0
8,11,42178,20385,1393,0.0
9,11,42038,20379,1387,0.0


### 2.6 Create FACT_CHARGING

In [36]:
# Process charging data
fact_charging = df_charging.copy()

# Rename charging_points to energy_consumed for consistency
fact_charging['energy_consumed'] = fact_charging['charging_points']

# Map location to location_id
fact_charging = fact_charging.merge(
    dim_location[['municipality_name', 'location_id']],
    left_on='location',
    right_on='municipality_name',
    how='left'
)

# Create datetime from year and month
fact_charging['datetime'] = pd.to_datetime(
    fact_charging['year'].astype(str) + '-' +
    fact_charging['month'].astype(str) + '-01'
)

# Map to month level time_id
fact_charging = fact_charging.merge(
    dim_time[['datetime', 'month_id']].drop_duplicates(),
    on='datetime',
    how='left'
)

# Select fact table columns
fact_charging = fact_charging[[
    'location_id',
    'month_id',
    'outlet_type',
    'energy_consumed'
]]

# Remove rows with missing key references
fact_charging = fact_charging.dropna(subset=['location_id', 'month_id', 'energy_consumed'])

logger.info(f"✓ FACT_CHARGING created: {len(fact_charging)} rows")
display(fact_charging.head(10))

2026-04-04 15:25:48,496 - INFO - ✓ FACT_CHARGING created: 3232 rows


,location_id,month_id,outlet_type,energy_consumed
0,2.0,1,Total,4.0
1,2.0,1,Mennekes,4.0
6,3.0,1,Total,2.0
7,3.0,1,Mennekes,2.0
12,4.0,1,Total,3.0
13,4.0,1,Mennekes,1.0
14,4.0,1,CHAdeMO,1.0
15,4.0,1,CCS,1.0
18,5.0,1,Total,14.0
19,5.0,1,Mennekes,10.0


### 2.7 Create FACT_POPULATION

In [37]:
# Process population data
fact_population = df_pop.copy()

# Remove rows with missing year or region
fact_population = fact_population.dropna(subset=['year', 'region_name'])

# Map region to location_id
fact_population = fact_population.merge(
    dim_location[['municipality_name', 'location_id']],
    left_on='region_name',
    right_on='municipality_name',
    how='left'
)

# Create datetime from year (01/01)
fact_population['datetime'] = pd.to_datetime(
    fact_population['year'].astype(str) + '-01-01'
)

# Map to year level time_id
fact_population = fact_population.merge(
    dim_time[['datetime', 'time_id']].drop_duplicates(),
    on='datetime',
    how='left'
)

# Convert measures to numeric
fact_population['density'] = pd.to_numeric(fact_population['density'], errors='coerce')
fact_population['growth_rate'] = pd.to_numeric(fact_population['growth_rate'], errors='coerce')

# Select fact table columns
fact_population = fact_population[[
    'location_id',
    'time_id',
    'density',
    'growth_rate'
]]

# Remove rows with missing key references
fact_population = fact_population.dropna(subset=['location_id', 'time_id'])

logger.info(f"✓ FACT_POPULATION created: {len(fact_population)} rows")
display(fact_population.head(10))

2026-04-04 15:25:53,350 - INFO - ✓ FACT_POPULATION created: 72 rows


,location_id,time_id,density,growth_rate
0,1,4,NaN,NaN
1,2,4,NaN,NaN
2,3,4,NaN,NaN
3,4,4,1284.0,NaN
4,5,4,1744.0,NaN
5,6,4,NaN,NaN
6,7,4,NaN,NaN
7,8,4,NaN,NaN
8,9,4,NaN,NaN
9,10,4,NaN,NaN


## Phase 3: Validation

Data quality checks and referential integrity

In [38]:
logger.info("=" * 60)
logger.info("PHASE 3: VALIDATION")
logger.info("=" * 60)

validation_report = []

def check_nulls(df, table_name, critical_columns):
    """Check for null values in critical columns"""
    null_count = df[critical_columns].isnull().sum()
    if null_count.sum() > 0:
        logger.warning(f"{table_name} - NULL values found:")
        for col, count in null_count[null_count > 0].items():
            logger.warning(f"  {col}: {count} nulls")
        validation_report.append(f"⚠ {table_name}: {null_count.sum()} null values")
    else:
        logger.info(f"✓ {table_name}: No null values in critical columns")
        validation_report.append(f"✓ {table_name}: No null values")

# Dimension validation
logger.info("\n--- DIMENSION VALIDATION ---")
check_nulls(dim_location, "DIM_LOCATION", ['location_id', 'municipality_code'])
check_nulls(dim_time, "DIM_TIME", ['time_id', 'year'])
check_nulls(dim_weather, "DIM_WEATHER", ['weather_id'])
check_nulls(dim_substations, "DIM_SUBSTATIONS", ['substation_id'])

# Fact validation
logger.info("\n--- FACT TABLE VALIDATION ---")
check_nulls(fact_consumption, "FACT_CONSUMPTION", ['location_id', 'hour_id', 'energy_consumed'])
check_nulls(fact_charging, "FACT_CHARGING", ['location_id', 'month_id', 'energy_consumed'])
check_nulls(fact_population, "FACT_POPULATION", ['location_id', 'time_id'])

2026-04-04 15:26:08,122 - INFO - ============================================================
2026-04-04 15:26:08,124 - INFO - PHASE 3: VALIDATION
2026-04-04 15:26:08,125 - INFO - ============================================================
2026-04-04 15:26:08,128 - INFO - 
--- DIMENSION VALIDATION ---
2026-04-04 15:26:08,132 - INFO - ✓ DIM_LOCATION: No null values in critical columns
2026-04-04 15:26:08,139 - INFO - ✓ DIM_TIME: No null values in critical columns
2026-04-04 15:26:08,143 - INFO - ✓ DIM_WEATHER: No null values in critical columns
2026-04-04 15:26:08,147 - INFO - ✓ DIM_SUBSTATIONS: No null values in critical columns
2026-04-04 15:26:08,149 - INFO - 
--- FACT TABLE VALIDATION ---
2026-04-04 15:26:08,189 - INFO - ✓ FACT_CONSUMPTION: No null values in critical columns
2026-04-04 15:26:08,193 - INFO - ✓ FACT_CHARGING: No null values in critical columns
2026-04-04 15:26:08,198 - INFO - ✓ FACT_POPULATION: No null values in critical columns


In [39]:
# Referential Integrity Checks
logger.info("\n--- REFERENTIAL INTEGRITY ---")

def check_foreign_keys(fact_df, dim_df, fact_col, dim_col, table_names):
    """Check if all FK values exist in dimension"""
    missing = fact_df[~fact_df[fact_col].isin(dim_df[dim_col])]
    if len(missing) > 0:
        logger.warning(f"{table_names[0]} → {table_names[1]}: {len(missing)} orphan records")
        validation_report.append(f"⚠ {table_names[0]} → {table_names[1]}: {len(missing)} orphan FK")
    else:
        logger.info(f"✓ {table_names[0]} → {table_names[1]}: FK integrity OK")
        validation_report.append(f"✓ {table_names[0]} → {table_names[1]}: FK OK ({len(fact_df)} rows)")

# Check consumption FKs
check_foreign_keys(fact_consumption, dim_location, 'location_id', 'location_id', 
                   ['FACT_CONSUMPTION', 'DIM_LOCATION'])
check_foreign_keys(fact_consumption, dim_time, 'hour_id', 'hour_id', 
                   ['FACT_CONSUMPTION', 'DIM_TIME'])
check_foreign_keys(fact_consumption, dim_weather, 'weather_id', 'weather_id', 
                   ['FACT_CONSUMPTION', 'DIM_WEATHER'])

# Check charging FKs
check_foreign_keys(fact_charging, dim_location, 'location_id', 'location_id', 
                   ['FACT_CHARGING', 'DIM_LOCATION'])
check_foreign_keys(fact_charging, dim_time, 'month_id', 'month_id', 
                   ['FACT_CHARGING', 'DIM_TIME'])

# Check population FKs
check_foreign_keys(fact_population, dim_location, 'location_id', 'location_id', 
                   ['FACT_POPULATION', 'DIM_LOCATION'])
check_foreign_keys(fact_population, dim_time, 'time_id', 'time_id', 
                   ['FACT_POPULATION', 'DIM_TIME'])

2026-04-04 15:26:12,241 - INFO - 
--- REFERENTIAL INTEGRITY ---
2026-04-04 15:26:12,269 - INFO - ✓ FACT_CONSUMPTION → DIM_LOCATION: FK integrity OK
2026-04-04 15:26:12,281 - INFO - ✓ FACT_CONSUMPTION → DIM_TIME: FK integrity OK
2026-04-04 15:26:12,292 - INFO - ✓ FACT_CONSUMPTION → DIM_WEATHER: FK integrity OK
2026-04-04 15:26:12,294 - INFO - ✓ FACT_CHARGING → DIM_LOCATION: FK integrity OK
2026-04-04 15:26:12,297 - INFO - ✓ FACT_CHARGING → DIM_TIME: FK integrity OK
2026-04-04 15:26:12,300 - INFO - ✓ FACT_POPULATION → DIM_LOCATION: FK integrity OK
2026-04-04 15:26:12,304 - INFO - ✓ FACT_POPULATION → DIM_TIME: FK integrity OK


In [40]:
# Summary Statistics
logger.info("\n--- SUMMARY STATISTICS ---")

summary = f"""
DIM_LOCATION:        {len(dim_location):>7} rows | PKs: {len(dim_location['location_id'].unique())} unique
DIM_TIME:            {len(dim_time):>7} rows | PKs: {len(dim_time['time_id'].unique())} unique
DIM_WEATHER:         {len(dim_weather):>7} rows | PKs: {len(dim_weather['weather_id'].unique())} unique
DIM_SUBSTATIONS:     {len(dim_substations):>7} rows | PKs: {len(dim_substations['substation_id'].unique())} unique

FACT_CONSUMPTION:    {len(fact_consumption):>7} rows | Energy: {fact_consumption['energy_consumed'].sum():.2f} kWh
FACT_CHARGING:       {len(fact_charging):>7} rows | Energy: {fact_charging['energy_consumed'].sum():.2f} kWh
FACT_POPULATION:     {len(fact_population):>7} rows | Locations: {len(fact_population['location_id'].unique())}
"""

logger.info(summary)
validation_report.append(summary)

2026-04-04 15:26:16,571 - INFO - 
--- SUMMARY STATISTICS ---
2026-04-04 15:26:16,580 - INFO - 
DIM_LOCATION:             18 rows | PKs: 18 unique
DIM_TIME:              45217 rows | PKs: 6 unique
DIM_WEATHER:            1520 rows | PKs: 1520 unique
DIM_SUBSTATIONS:        8412 rows | PKs: 8412 unique

FACT_CONSUMPTION:    1027232 rows | Energy: 29865932.19 kWh
FACT_CHARGING:          3232 rows | Energy: 231884.00 kWh
FACT_POPULATION:          72 rows | Locations: 18



## Phase 4: Loading

Export to CSV files

In [41]:
logger.info("=" * 60)
logger.info("PHASE 4: LOADING")
logger.info("=" * 60)

# Export Dimensions
logger.info("\nExporting dimension tables...")
dim_location.to_csv(os.path.join(DIMS_DIR, 'dim_location.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_location.csv")

dim_time.to_csv(os.path.join(DIMS_DIR, 'dim_time.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_time.csv")

dim_weather.to_csv(os.path.join(DIMS_DIR, 'dim_weather.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_weather.csv")

dim_substations.to_csv(os.path.join(DIMS_DIR, 'dim_substations.csv'), index=False)
logger.info(f"✓ {DIMS_DIR}/dim_substations.csv")

# Export Facts
logger.info("\nExporting fact tables...")
fact_consumption.to_csv(os.path.join(FACTS_DIR, 'fact_consumption.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_consumption.csv")

fact_charging.to_csv(os.path.join(FACTS_DIR, 'fact_charging.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_charging.csv")

fact_population.to_csv(os.path.join(FACTS_DIR, 'fact_population.csv'), index=False)
logger.info(f"✓ {FACTS_DIR}/fact_population.csv")

logger.info("\n✓ All tables exported successfully")

2026-04-04 15:26:28,294 - INFO - ============================================================
2026-04-04 15:26:28,296 - INFO - PHASE 4: LOADING
2026-04-04 15:26:28,298 - INFO - ============================================================
2026-04-04 15:26:28,299 - INFO - 
Exporting dimension tables...
2026-04-04 15:26:28,305 - INFO - ✓ ./dw_output\dimensions/dim_location.csv
2026-04-04 15:26:28,576 - INFO - ✓ ./dw_output\dimensions/dim_time.csv
2026-04-04 15:26:28,600 - INFO - ✓ ./dw_output\dimensions/dim_weather.csv
2026-04-04 15:26:28,652 - INFO - ✓ ./dw_output\dimensions/dim_substations.csv
2026-04-04 15:26:28,653 - INFO - 
Exporting fact tables...
2026-04-04 15:26:30,728 - INFO - ✓ ./dw_output\facts/fact_consumption.csv
2026-04-04 15:26:30,739 - INFO - ✓ ./dw_output\facts/fact_charging.csv
2026-04-04 15:26:30,743 - INFO - ✓ ./dw_output\facts/fact_population.csv
2026-04-04 15:26:30,745 - INFO - 
✓ All tables exported successfully


In [42]:
# Generate Validation Report
report_path = os.path.join(LOGS_DIR, 'etl_validation_report.txt')

with open(report_path, 'w') as f:
    f.write("="*60 + "\n")
    f.write("DATA WAREHOUSE ETL VALIDATION REPORT\n")
    f.write("="*60 + "\n\n")
    
    f.write(f"Execution Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("VALIDATION RESULTS:\n")
    f.write("-" * 60 + "\n")
    for item in validation_report:
        f.write(str(item) + "\n")
    
    f.write("\n" + "-" * 60 + "\n")
    f.write(f"ETL EXECUTION: SUCCESS\n")
    f.write(f"Output Directory: {OUTPUT_DIR}\n")

logger.info(f"✓ Validation report saved: {report_path}")
logger.info("="*60)
logger.info("ETL PROCESS COMPLETED SUCCESSFULLY")
logger.info("="*60)

2026-04-04 15:26:36,051 - INFO - ✓ Validation report saved: ./dw_output\logs\etl_validation_report.txt
2026-04-04 15:26:36,053 - INFO - ============================================================
2026-04-04 15:26:36,054 - INFO - ETL PROCESS COMPLETED SUCCESSFULLY
2026-04-04 15:26:36,056 - INFO - ============================================================


## Summary

### Dimensional Bus Matrix Implementation

| Data Mart | Star | Dimensions Used |
|-----------|------|------------------|
| Energy | Consumption | Location, Time, Weather, Substations |
| Mobility | Charging | Location, Time |
| Demographics | Population | Location, Time |

### Output Files

**Dimensions:**
- `dimensions/dim_location.csv` - Geographic hierarchy
- `dimensions/dim_time.csv` - Multi-level temporal dimension
- `dimensions/dim_weather.csv` - Daily weather conditions
- `dimensions/dim_substations.csv` - Electrical infrastructure

**Facts:**
- `facts/fact_consumption.csv` - Hourly energy consumption
- `facts/fact_charging.csv` - Monthly EV charging activity
- `facts/fact_population.csv` - Yearly demographic indicators

### Next Steps

1. **Load to Database**: Create SQL schema and load CSVs to database
2. **Add Indexes**: Create indexes on PK, FK, and commonly filtered columns
3. **OLAP Cubes**: Build aggregated views for reporting
4. **Test Queries**: Run dimensional analysis queries
5. **Documentation**: Create data dictionary and business glossary

In [51]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables from .env file (located in parent directory)
env_path = Path.cwd().parent / '.env'
if not env_path.exists():
    env_path = Path.cwd() / '.env'
load_dotenv(env_path)

# Step 1: Load to Database - Connect to MySQL and load data

logger.info("=" * 80)
logger.info("STEP 1: LOAD TO DATABASE - Connecting to MySQL and loading data")
logger.info("=" * 80)

# ============================================================================
# MYSQL CONFIGURATION - Load from .env file
# ============================================================================
MYSQL_HOST = os.getenv('MYSQL_HOST', 'localhost')
MYSQL_PORT = int(os.getenv('MYSQL_PORT', 3306))
MYSQL_USER = os.getenv('MYSQL_USER', 'root')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD', '')
MYSQL_DATABASE = os.getenv('MYSQL_DATABASE', 'data_warehouse')

logger.info(f"\nMySQL Connection Parameters:")
logger.info(f"  Host: {MYSQL_HOST}")
logger.info(f"  Port: {MYSQL_PORT}")
logger.info(f"  Database: {MYSQL_DATABASE}")
logger.info(f"  User: {MYSQL_USER}")

# Step 1: Connect to MySQL and create database if needed
logger.info("\nAttempting to connect to MySQL server...")

# First, connect to MySQL without specifying a database
root_connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}'

try:
    # Connect to MySQL server
    root_engine = create_engine(root_connection_string, echo=False)
    
    with root_engine.connect() as connection:
        connection.execute(text("SELECT 1"))
    logger.info("Connected to MySQL server successfully!")
    
    # Create database if it doesn't exist
    logger.info(f"\nChecking if database '{MYSQL_DATABASE}' exists...")
    with root_engine.begin() as connection:
        connection.execute(text(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}"))
    logger.info(f"Database '{MYSQL_DATABASE}' is ready!")
    
    # Now connect to the specific database
    connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
    engine = create_engine(connection_string, echo=False)
    
    logger.info("\nAttempting to connect to data warehouse database...")
    with engine.connect() as test_connection:
        test_connection.execute(text("SELECT 1"))
    logger.info("Connected to data warehouse database successfully!")
    
except Exception as e:
    logger.error(f"Failed to connect to MySQL")
    logger.error(f"Error: {e}")
    logger.info("\nPlease verify your MySQL connection settings in the .env file:")
    logger.info(f"  - Host: {MYSQL_HOST}:{MYSQL_PORT}")
    logger.info(f"  - User: {MYSQL_USER}")
    logger.info("\nUpdate the MYSQL_USER and MYSQL_PASSWORD in .env if needed.")
    raise

# Load dimensions to database
logger.info("\n" + "-" * 80)
logger.info("Loading dimension tables to database...")
logger.info("-" * 80)

try:
    dim_location.to_sql('dim_location', engine, if_exists='replace', index=False)
    logger.info(f"  dim_location: {len(dim_location):,} rows loaded")

    dim_time.to_sql('dim_time', engine, if_exists='replace', index=False)
    logger.info(f"  dim_time: {len(dim_time):,} rows loaded")

    dim_weather.to_sql('dim_weather', engine, if_exists='replace', index=False)
    logger.info(f"  dim_weather: {len(dim_weather):,} rows loaded")

    dim_substations.to_sql('dim_substations', engine, if_exists='replace', index=False)
    logger.info(f"  dim_substations: {len(dim_substations):,} rows loaded")
    
except Exception as e:
    logger.error(f"Failed to load dimensions: {e}")
    raise

# Load facts to database
logger.info("\n" + "-" * 80)
logger.info("Loading fact tables to database...")
logger.info("-" * 80)

try:
    fact_consumption.to_sql('fact_consumption', engine, if_exists='replace', index=False)
    logger.info(f" fact_consumption: {len(fact_consumption):,} rows loaded")

    fact_charging.to_sql('fact_charging', engine, if_exists='replace', index=False)
    logger.info(f" fact_charging: {len(fact_charging):,} rows loaded")

    fact_population.to_sql('fact_population', engine, if_exists='replace', index=False)
    logger.info(f" fact_population: {len(fact_population):,} rows loaded")
    
except Exception as e:
    logger.error(f"Failed to load facts: {e}")
    raise

# Verify tables were created
logger.info("\n" + "-" * 80)
logger.info("Verifying tables in database...")
logger.info("-" * 80)

try:
    with engine.connect() as connection:
        result = connection.execute(text(f"""
            SELECT TABLE_NAME 
            FROM INFORMATION_SCHEMA.TABLES 
            WHERE TABLE_SCHEMA = '{MYSQL_DATABASE}'
            ORDER BY TABLE_NAME
        """))
        tables = [row[0] for row in result]
        logger.info(f"\nTables created in {MYSQL_DATABASE} database:")
        for table in tables:
            logger.info(f"  {table}")
        logger.info(f"\nTotal tables: {len(tables)}")
        
except Exception as e:
    logger.error(f"Failed to verify tables: {e}")
    raise

logger.info("\n" + "=" * 80)
logger.info("STEP 1 COMPLETED SUCCESSFULLY!")
logger.info("=" * 80)

2026-04-04 15:40:32,599 - INFO - ================================================================================
2026-04-04 15:40:32,601 - INFO - STEP 1: LOAD TO DATABASE - Connecting to MySQL and loading data
2026-04-04 15:40:32,602 - INFO - ================================================================================
2026-04-04 15:40:32,604 - INFO - 
MySQL Connection Parameters:
2026-04-04 15:40:32,606 - INFO -   Host: localhost
2026-04-04 15:40:32,608 - INFO -   Port: 3306
2026-04-04 15:40:32,610 - INFO -   Database: data_warehouse
2026-04-04 15:40:32,613 - INFO -   User: root
2026-04-04 15:40:32,614 - INFO - 
Attempting to connect to MySQL server...
2026-04-04 15:40:32,623 - INFO - Connected to MySQL server successfully!
2026-04-04 15:40:32,624 - INFO - 
Checking if database 'data_warehouse' exists...
2026-04-04 15:40:32,629 - INFO - Database 'data_warehouse' is ready!
2026-04-04 15:40:32,631 - INFO - 
Attempting to connect to data warehouse database...
2026-04-04 15:40:32,649 

In [52]:
# Step 2: Add Indexes - Create indexes for query optimization

logger.info("\n" + "=" * 80)
logger.info("STEP 2: ADD INDEXES - Creating indexes for query optimization")
logger.info("=" * 80)

with engine.begin() as connection:
    # Create indexes on dimension tables (Primary Keys)
    logger.info("\nCreating indexes on Dimension Tables...")
    
    index_commands = [
        # Dimension indexes (Primary Keys)
        "CREATE INDEX idx_dim_location_pk ON DIM_LOCATION(location_id)",
        "CREATE INDEX idx_dim_time_pk ON DIM_TIME(time_id)",
        "CREATE INDEX idx_dim_weather_pk ON DIM_WEATHER(weather_id)",
        "CREATE INDEX idx_dim_substations_pk ON DIM_SUBSTATIONS(substation_id)",
        
        # Fact table indexes (Foreign Keys)
        "CREATE INDEX idx_fact_consumption_location ON FACT_CONSUMPTION(location_id)",
        "CREATE INDEX idx_fact_consumption_hour ON FACT_CONSUMPTION(hour_id)",
        "CREATE INDEX idx_fact_consumption_weather ON FACT_CONSUMPTION(weather_id)",
        "CREATE INDEX idx_fact_consumption_substation ON FACT_CONSUMPTION(substation_id)",
        
        "CREATE INDEX idx_fact_charging_location ON FACT_CHARGING(location_id)",
        "CREATE INDEX idx_fact_charging_month ON FACT_CHARGING(month_id)",
        
        "CREATE INDEX idx_fact_population_location ON FACT_POPULATION(location_id)",
        "CREATE INDEX idx_fact_population_year ON FACT_POPULATION(time_id)",
        
        # Commonly filtered columns in dimensions
        "CREATE INDEX idx_dim_location_municipality ON DIM_LOCATION(municipality_name)",
        "CREATE INDEX idx_dim_location_region ON DIM_LOCATION(region_name)",
        
        "CREATE INDEX idx_dim_time_date ON DIM_TIME(date)",
        "CREATE INDEX idx_dim_time_year ON DIM_TIME(year)",
        "CREATE INDEX idx_dim_time_month ON DIM_TIME(month)",
        
        "CREATE INDEX idx_dim_substations_municipality ON DIM_SUBSTATIONS(municipality_name)",
    ]
    
    for idx_cmd in index_commands:
        try:
            connection.execute(text(idx_cmd))
            logger.info(f"  ✓ {idx_cmd.split('ON')[1].strip()}")
        except Exception as e:
            # MySQL throws error if index already exists without IF NOT EXISTS
            # This is expected behavior on re-runs
            if "Duplicate key name" in str(e) or "already exists" in str(e):
                logger.info(f"  ℹ Index already exists: {idx_cmd.split('ON')[1].strip()}")
            else:
                logger.warning(f"  ✗ {str(e)}")
    
    logger.info("✓ Step 2 completed successfully!")

2026-04-04 15:41:11,084 - INFO - 
2026-04-04 15:41:11,088 - INFO - STEP 2: ADD INDEXES - Creating indexes for query optimization
2026-04-04 15:41:11,093 - INFO - ================================================================================
2026-04-04 15:41:11,096 - INFO - 
Creating indexes on Dimension Tables...


2026-04-04 15:41:11,164 - INFO -   ✓ DIM_LOCATI
2026-04-04 15:41:11,385 - INFO -   ✓ DIM_TIME(time_id)
2026-04-04 15:41:11,453 - INFO -   ✓ DIM_WEATHER(weather_id)
2026-04-04 15:41:11,559 - INFO -   ✓ DIM_SUBSTATI
2026-04-04 15:41:13,664 - INFO -   ✓ FACT_C
2026-04-04 15:41:15,821 - INFO -   ✓ FACT_C
2026-04-04 15:41:17,985 - INFO -   ✓ FACT_C
2026-04-04 15:41:17,987 - WARNING -   ✗ (pymysql.err.OperationalError) (1072, "Key column 'substation_id' doesn't exist in table")
[SQL: CREATE INDEX idx_fact_consumption_substation ON FACT_CONSUMPTION(substation_id)]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
2026-04-04 15:41:18,041 - INFO -   ✓ FACT_CHARGING(location_id)
2026-04-04 15:41:18,093 - INFO -   ✓ FACT_CHARGING(month_id)
2026-04-04 15:41:18,127 - INFO -   ✓ FACT_POPULATI
2026-04-04 15:41:18,180 - INFO -   ✓ FACT_POPULATI
2026-04-04 15:41:18,183 - WARNING -   ✗ (pymysql.err.OperationalError) (1170, "BLOB/TEXT column 'municipality_name' used in key specification withou

In [54]:
# Step 3: OLAP Cubes - Build aggregated views for reporting

logger.info("\n" + "=" * 80)
logger.info("STEP 3: OLAP CUBES - Creating aggregated views for reporting")
logger.info("=" * 80)

# Create aggregated views for OLAP analysis
aggregated_views = []

logger.info("\nCreating aggregated views...")

# View 1: Daily Energy Consumption by Location
daily_consumption = fact_consumption.groupby(['day_id', 'location_id']).agg({
    'energy_consumed': 'sum',
    'hour_id': 'count'
}).rename(columns={'hour_id': 'hours_recorded'}).reset_index()
daily_consumption.to_sql('view_daily_consumption_by_location', engine, if_exists='replace', index=False)
logger.info(f"  ✓ view_daily_consumption_by_location: {len(daily_consumption)} rows")
aggregated_views.append(('view_daily_consumption_by_location', len(daily_consumption)))

# View 2: Monthly EV Charging by District
monthly_charging_district = fact_charging.merge(
    dim_location[['location_id', 'district_name']], on='location_id'
).groupby(['month_id', 'district_name']).agg({
    'energy_consumed': 'mean',
    'location_id': 'count'
}).rename(columns={'location_id': 'locations_count'}).reset_index()
monthly_charging_district.to_sql('view_monthly_charging_by_district', engine, if_exists='replace', index=False)
logger.info(f"  ✓ view_monthly_charging_by_district: {len(monthly_charging_district)} rows")
aggregated_views.append(('view_monthly_charging_by_district', len(monthly_charging_district)))

# View 3: Weather Impact on Energy Consumption
weather_consumption = fact_consumption.merge(
    dim_weather[['weather_id', 'avg_temperature', 'precipitation']], on='weather_id'
).groupby(['avg_temperature', 'precipitation']).agg({
    'energy_consumed': ['sum', 'mean', 'count']
}).reset_index()
weather_consumption.columns = ['avg_temperature', 'precipitation', 'total_consumption', 'avg_consumption', 'records']
weather_consumption.to_sql('view_weather_consumption_impact', engine, if_exists='replace', index=False)
logger.info(f"  ✓ view_weather_consumption_impact: {len(weather_consumption)} rows")
aggregated_views.append(('view_weather_consumption_impact', len(weather_consumption)))

# View 4: Yearly Demographic Trends
yearly_demographics = fact_population.merge(
    dim_location[['location_id', 'municipality_name']], on='location_id'
).merge(
    dim_time[['time_id', 'year']], on='time_id'
).groupby(['year', 'municipality_name']).agg({
    'density': 'mean',
    'growth_rate': 'mean'
}).reset_index()
yearly_demographics.to_sql('view_yearly_demographic_trends', engine, if_exists='replace', index=False)
logger.info(f"  ✓ view_yearly_demographic_trends: {len(yearly_demographics)} rows")
aggregated_views.append(('view_yearly_demographic_trends', len(yearly_demographics)))

logger.info(f"\nCreated {len(aggregated_views)} aggregated views")
logger.info("✓ Step 3 completed successfully!")


2026-04-04 15:41:55,691 - INFO - 
2026-04-04 15:41:55,692 - INFO - STEP 3: OLAP CUBES - Creating aggregated views for reporting
2026-04-04 15:41:55,695 - INFO - ================================================================================
2026-04-04 15:41:55,697 - INFO - 
Creating aggregated views...
2026-04-04 15:41:56,009 - INFO -   ✓ view_daily_consumption_by_location: 6205 rows
2026-04-04 15:41:56,137 - INFO -   ✓ view_monthly_charging_by_district: 864 rows
2026-04-04 15:41:56,376 - INFO -   ✓ view_weather_consumption_impact: 330 rows
2026-04-04 15:41:56,548 - INFO -   ✓ view_yearly_demographic_trends: 72 rows
2026-04-04 15:41:56,550 - INFO - 
Created 4 aggregated views
2026-04-04 15:41:56,552 - INFO - ✓ Step 3 completed successfully!


In [55]:
# Step 4: Test Queries - Run dimensional analysis queries

logger.info("\n" + "=" * 80)
logger.info("STEP 4: TEST QUERIES - Running dimensional analysis queries")
logger.info("=" * 80)

query_results = []

# Query 1: Total consumption by district
query1 = """
SELECT 
    dl.district_name,
    SUM(fc.energy_consumed) as total_consumption,
    AVG(fc.energy_consumed) as avg_consumption,
    COUNT(*) as records
FROM FACT_CONSUMPTION fc
JOIN DIM_LOCATION dl ON fc.location_id = dl.location_id
GROUP BY dl.district_name
ORDER BY total_consumption DESC;
"""
logger.info("\nQuery 1: Total consumption by district")
result1 = pd.read_sql_query(query1, engine)
logger.info(f"\nResults ({len(result1)} districts):")
print(result1.to_string())
query_results.append(('Q1_Consumption_by_District', result1))

# Query 2: Peak consumption hours
query2 = """
SELECT 
    dt.hour,
    AVG(fc.energy_consumed) as avg_consumption,
    MAX(fc.energy_consumed) as max_consumption,
    COUNT(*) as records
FROM FACT_CONSUMPTION fc
JOIN DIM_TIME dt ON fc.hour_id = dt.hour_id
GROUP BY dt.hour
ORDER BY avg_consumption DESC;
"""
logger.info("\n" + "-" * 80)
logger.info("Query 2: Peak consumption hours")
result2 = pd.read_sql_query(query2, engine)
logger.info(f"\nTop 5 peak hours:")
print(result2.head(5).to_string())
query_results.append(('Q2_Peak_Hours', result2))

# Query 3: Weather correlation with consumption
query3 = """
SELECT 
    dw.avg_temperature as temperature,
    ROUND(dw.precipitation, 2) as precipitation,
    AVG(fc.energy_consumed) as avg_consumption,
    COUNT(*) as sample_size
FROM FACT_CONSUMPTION fc
JOIN DIM_WEATHER dw ON fc.weather_id = dw.weather_id
GROUP BY dw.avg_temperature, dw.precipitation
ORDER BY avg_consumption DESC
LIMIT 10;
"""
logger.info("\n" + "-" * 80)
logger.info("Query 3: Weather correlation with consumption")
result3 = pd.read_sql_query(query3, engine)
logger.info(f"\nTop conditions:")
print(result3.to_string())
query_results.append(('Q3_Weather_Correlation', result3))

# Query 4: Charging activity by municipality and outlet type
query4 = """
SELECT 
    dl.municipality_name,
    fch.outlet_type,
    SUM(fch.energy_consumed) as total_energy,
    AVG(fch.energy_consumed) as avg_energy,
    COUNT(*) as records
FROM FACT_CHARGING fch
JOIN DIM_LOCATION dl ON fch.location_id = dl.location_id
GROUP BY dl.municipality_name, fch.outlet_type
ORDER BY total_energy DESC
LIMIT 10;
"""
logger.info("\n" + "-" * 80)
logger.info("Query 4: Top charging municipalities by outlet type")
result4 = pd.read_sql_query(query4, engine)
logger.info(f"\nTop 10 results:")
print(result4.to_string())
query_results.append(('Q4_Top_Charging_Activity', result4))

# Query 5: Demographics statistics by district
query5 = """
SELECT 
    dl.district_name,
    COUNT(*) as location_count,
    AVG(fp.density) as avg_density,
    AVG(fp.growth_rate) as avg_growth_rate
FROM FACT_POPULATION fp
JOIN DIM_LOCATION dl ON fp.location_id = dl.location_id
GROUP BY dl.district_name
ORDER BY avg_density DESC;
"""
logger.info("\n" + "-" * 80)
logger.info("Query 5: Demographics statistics by district")
result5 = pd.read_sql_query(query5, engine)
logger.info(f"\nResults ({len(result5)} districts):")
print(result5.to_string())
query_results.append(('Q5_Demographics_Statistics', result5))

logger.info("\n" + "=" * 80)
logger.info(f"✓ Step 4 completed successfully! ({len(query_results)} queries executed)")

2026-04-04 15:42:00,301 - INFO - 
2026-04-04 15:42:00,303 - INFO - STEP 4: TEST QUERIES - Running dimensional analysis queries
2026-04-04 15:42:00,304 - INFO - ================================================================================
2026-04-04 15:42:00,306 - INFO - 
Query 1: Total consumption by district
2026-04-04 15:42:15,185 - INFO - 
Results (17 districts):
2026-04-04 15:42:15,189 - INFO - 
--------------------------------------------------------------------------------
2026-04-04 15:42:15,191 - INFO - Query 2: Peak consumption hours


           district_name  total_consumption  avg_consumption  records
0                  Porto         6788796.08       116.354096    58346
1      Vila Nova de Gaia         5984893.29        48.850689   122514
2             Matosinhos         4981788.82       149.972570    33218
3                   Maia         2294091.68        30.991606    74023
4               Gondomar         1869293.35        37.777239    49482
5                Valongo         1687692.88        50.687556    33296
6          Vila do Conde         1470714.36        16.766774    87716
7   Santa Maria da Feira         1277528.04         6.944333   183967
8        Póvoa de Varzim          856565.85        17.501652    48942
9                Paredes          585036.94         9.401960    62225
10           Santo Tirso          502330.66         6.710178    74861
11   São João da Madeira          481907.08        55.005945     8761
12                 Trofa          356237.33         8.712728    40887
13               Esp

2026-04-04 15:42:21,904 - INFO - 
Top 5 peak hours:
2026-04-04 15:42:21,906 - INFO - 
--------------------------------------------------------------------------------
2026-04-04 15:42:21,906 - INFO - Query 3: Weather correlation with consumption


   hour  avg_consumption  max_consumption  records
0    12        44.941515           917.37    42867
1    13        44.842977           976.23    42777
2    16        44.108575           866.57    42863
3    15        43.524817          1074.35    42828
4    18        43.429846          1053.98    42801


2026-04-04 15:42:26,249 - INFO - 
Top conditions:
2026-04-04 15:42:26,252 - INFO - 
--------------------------------------------------------------------------------
2026-04-04 15:42:26,252 - INFO - Query 4: Top charging municipalities by outlet type
2026-04-04 15:42:26,303 - INFO - 
Top 10 results:
2026-04-04 15:42:26,306 - INFO - 
--------------------------------------------------------------------------------
2026-04-04 15:42:26,307 - INFO - Query 5: Demographics statistics by district
2026-04-04 15:42:26,313 - INFO - 
Results (18 districts):
2026-04-04 15:42:26,317 - INFO - 
2026-04-04 15:42:26,319 - INFO - ✓ Step 4 completed successfully! (5 queries executed)


   temperature  precipitation  avg_consumption  sample_size
0        10.10           17.2        41.778531         2846
1        10.35           20.7        39.843397         2829
2         8.65           40.3        39.544195         2801
3         9.65            0.0        39.450856         2933
4        11.15           62.2        39.261879         2800
5         7.95           19.7        38.793432         2809
6         6.55           24.3        38.687381         2841
7         7.90            0.0        38.598454         2937
8         8.95           50.8        37.260381         2679
9        11.05           21.5        37.119956         2934
             municipality_name outlet_type  total_energy   avg_energy  records
0  Área Metropolitana do Porto       Total       57971.0  1207.729167       48
1  Área Metropolitana do Porto    Mennekes       37820.0   787.916667       48
2                        Porto       Total       15764.0   328.416667       48
3  Área Metropolitana do

In [57]:
# Step 5: Documentation - Create data dictionary and business glossary

logger.info("\n" + "=" * 80)
logger.info("STEP 5: DOCUMENTATION - Creating data dictionary and business glossary")
logger.info("=" * 80)

# Initialize documentation
data_dictionary = []
business_glossary = []

# Data Dictionary - Dimensions
logger.info("\nGenerating data dictionary for dimensions...")

dimensions_doc = {
    'DIM_LOCATION': {
        'description': 'Geographic hierarchy of Porto Metropolitan Area',
        'columns': dim_location.columns.tolist(),
        'row_count': len(dim_location),
        'grain': 'One row per municipality with geographic hierarchies'
    },
    'DIM_TIME': {
        'description': 'Multi-level temporal dimension with hourly granularity',
        'columns': dim_time.columns.tolist(),
        'row_count': len(dim_time),
        'grain': 'One row per hour from min_date to max_date'
    },
    'DIM_WEATHER': {
        'description': 'Daily weather conditions from two monitoring stations',
        'columns': dim_weather.columns.tolist(),
        'row_count': len(dim_weather),
        'grain': 'One row per day with aggregated weather metrics'
    },
    'DIM_SUBSTATIONS': {
        'description': 'Electrical infrastructure substation details',
        'columns': dim_substations.columns.tolist(),
        'row_count': len(dim_substations),
        'grain': 'One row per unique substation'
    }
}

# Data Dictionary - Facts
facts_doc = {
    'FACT_CONSUMPTION': {
        'description': 'Hourly energy consumption by municipality',
        'columns': fact_consumption.columns.tolist(),
        'row_count': len(fact_consumption),
        'grain': 'One row per municipality per hour',
        'measures': ['energy_consumed']
    },
    'FACT_CHARGING': {
        'description': 'Monthly EV charging activity and outlet types',
        'columns': fact_charging.columns.tolist(),
        'row_count': len(fact_charging),
        'grain': 'One row per municipality per month per outlet type',
        'measures': ['energy_consumed']
    },
    'FACT_POPULATION': {
        'description': 'Yearly demographic indicators by municipality',
        'columns': fact_population.columns.tolist(),
        'row_count': len(fact_population),
        'grain': 'One row per municipality per year',
        'measures': ['density', 'growth_rate']
    }
}

# Create comprehensive documentation
documentation_content = f"""
# DATA WAREHOUSE DOCUMENTATION
## Porto Metropolitan Area - Energy, Mobility & Demographics Data Warehouse

Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## 1. EXECUTIVE SUMMARY

This data warehouse integrates energy consumption, electric vehicle charging infrastructure, and demographic data for the Porto Metropolitan Area. The schema follows a star topology with conformed dimensions enabling analysis across multiple data marts.

### Data Warehouse Dimensions
- **Geographic**: Municipality-level analysis with regional/district hierarchies
- **Temporal**: Hourly granularity with multi-level hierarchies (hour, day, month, year)
- **Weather**: Daily weather conditions from two monitoring stations
- **Infrastructure**: Electrical distribution substation details

### Data Warehouse Facts
- **FACT_CONSUMPTION**: {len(fact_consumption):,} hourly energy consumption records
- **FACT_CHARGING**: {len(fact_charging):,} EV charging activity records
- **FACT_POPULATION**: {len(fact_population):,} yearly demographic records

---

## 2. DIMENSIONAL BUS MATRIX

| Data Mart | Fact Table | Dimensions |
|-----------|-----------|-----------|
| Energy | FACT_CONSUMPTION | Location, Time, Weather, Substations |
| Mobility | FACT_CHARGING | Location, Time |
| Demographics | FACT_POPULATION | Location, Time |

---

## 3. DIMENSION DOCUMENTATION

### DIM_LOCATION
**Purpose**: {dimensions_doc['DIM_LOCATION']['description']}
**Grain**: {dimensions_doc['DIM_LOCATION']['grain']}
**Row Count**: {dimensions_doc['DIM_LOCATION']['row_count']:,}

**Columns**:
"""

for col in dimensions_doc['DIM_LOCATION']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

### DIM_TIME
**Purpose**: {dimensions_doc['DIM_TIME']['description']}
**Grain**: {dimensions_doc['DIM_TIME']['grain']}
**Row Count**: {dimensions_doc['DIM_TIME']['row_count']:,}
**Date Range**: {dim_time['date'].min()} to {dim_time['date'].max()}

**Columns**:
"""

for col in dimensions_doc['DIM_TIME']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

### DIM_WEATHER
**Purpose**: {dimensions_doc['DIM_WEATHER']['description']}
**Grain**: {dimensions_doc['DIM_WEATHER']['grain']}
**Row Count**: {dimensions_doc['DIM_WEATHER']['row_count']:,}

**Columns**:
"""

for col in dimensions_doc['DIM_WEATHER']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

### DIM_SUBSTATIONS
**Purpose**: {dimensions_doc['DIM_SUBSTATIONS']['description']}
**Grain**: {dimensions_doc['DIM_SUBSTATIONS']['grain']}
**Row Count**: {dimensions_doc['DIM_SUBSTATIONS']['row_count']:,}

**Columns**:
"""

for col in dimensions_doc['DIM_SUBSTATIONS']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

## 4. FACT TABLE DOCUMENTATION

### FACT_CONSUMPTION
**Purpose**: {facts_doc['FACT_CONSUMPTION']['description']}
**Grain**: {facts_doc['FACT_CONSUMPTION']['grain']}
**Row Count**: {facts_doc['FACT_CONSUMPTION']['row_count']:,}
**Measures**: {', '.join(facts_doc['FACT_CONSUMPTION']['measures'])}

**Columns**:
"""

for col in facts_doc['FACT_CONSUMPTION']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

### FACT_CHARGING
**Purpose**: {facts_doc['FACT_CHARGING']['description']}
**Grain**: {facts_doc['FACT_CHARGING']['grain']}
**Row Count**: {facts_doc['FACT_CHARGING']['row_count']:,}
**Measures**: {', '.join(facts_doc['FACT_CHARGING']['measures'])}

**Columns**:
"""

for col in facts_doc['FACT_CHARGING']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

### FACT_POPULATION
**Purpose**: {facts_doc['FACT_POPULATION']['description']}
**Grain**: {facts_doc['FACT_POPULATION']['grain']}
**Row Count**: {facts_doc['FACT_POPULATION']['row_count']:,}
**Measures**: {', '.join(facts_doc['FACT_POPULATION']['measures'])}

**Columns**:
"""

for col in facts_doc['FACT_POPULATION']['columns']:
    documentation_content += f"\n- `{col}`"

documentation_content += f"""

---

## 5. BUSINESS GLOSSARY

### Energy & Infrastructure Terms
- **Energy Consumed**: Electrical power consumption measured in kilowatt-hours (kWh)
- **Substation**: Electrical distribution point for power transmission
- **Installation Code**: Unique identifier for electrical infrastructure
- **District**: Administrative district in Porto Metropolitan Area

### Mobility Terms
- **Outlet Type**: Classification of EV charging outlets (e.g., Total, DC, AC)
- **Charging Energy**: Energy provided by charging infrastructure

### Geographic Terms
- **Municipality**: Administrative subdivision of Porto Metropolitan Area
- **District**: Grouping of municipalities for analysis
- **Parish**: Sub-division of municipality for geographic analysis

### Demographic Terms
- **Density**: Population density (people per square km)
- **Growth Rate**: Year-over-year population growth percentage

### Temporal Terms
- **Hour ID**: Unique identifier for each hour in the temporal dimension
- **Day ID**: Unique identifier for each day in the temporal dimension
- **Month ID**: Unique identifier for each month in the temporal dimension
- **Time ID**: Surrogate key for time dimension records

### Weather Terms
- **Average Temperature**: Mean temperature for the day (°C)
- **Minimum Temperature**: Lowest temperature recorded for the day (°C)
- **Maximum Temperature**: Highest temperature recorded for the day (°C)
- **Precipitation**: Total rainfall for the day (mm)
- **Wind Speed**: Average wind speed for the day (km/h)
- **Pressure**: Atmospheric pressure (hPa)

---

## 6. AGGREGATED VIEWS

The following views have been created for reporting:
- `VIEW_DAILY_CONSUMPTION_BY_LOCATION`: Daily energy totals by municipality
- `VIEW_MONTHLY_CHARGING_BY_DISTRICT`: Monthly charging activity by district
- `VIEW_WEATHER_CONSUMPTION_IMPACT`: Consumption patterns by weather conditions
- `VIEW_YEARLY_DEMOGRAPHIC_TRENDS`: Demographic trends by municipality

---

## 7. DATA QUALITY METRICS

**Completeness**:
- DIM_LOCATION: 100% ({len(dim_location)} municipalities)
- DIM_TIME: {(~dim_time['datetime'].isna()).sum() / len(dim_time) * 100:.1f}% ({(~dim_time['datetime'].isna()).sum():,} records)
- DIM_WEATHER: {(~dim_weather['weather_id'].isna()).sum() / len(dim_weather) * 100:.1f}% ({(~dim_weather['weather_id'].isna()).sum():,} records)
- DIM_SUBSTATIONS: {(~dim_substations['substation_id'].isna()).sum() / len(dim_substations) * 100:.1f}% ({(~dim_substations['substation_id'].isna()).sum():,} records)

**Grain Validation**:
- FACT_CONSUMPTION: {len(fact_consumption):,} hourly records (expected granularity maintained)
- FACT_CHARGING: {len(fact_charging):,} records (expected granularity maintained)
- FACT_POPULATION: {len(fact_population):,} yearly records (expected granularity maintained)

---

## 8. DATABASE ARTIFACTS

**Database Location**: MySQL - {MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}
**Database Type**: MySQL 8.0+
**Database User**: {MYSQL_USER}

**Tables Created**: 
- 7 (4 dimensions + 3 facts)

**Indexes Created**: 17
- Primary key indexes for all dimensions
- Foreign key indexes for all facts
- Commonly queried column indexes

---

## 9. ETL PROCESS SUMMARY

**Extract Phase**: 4 source datasets processed
**Transform Phase**: Data cleaning, aggregation, and standardization
**Load Phase**: Data loaded to MySQL database with full integrity
**Validation Phase**: 7 data quality checks performed
**Reporting Phase**: Aggregated views created

---

## 10. CONTACT & SUPPORT

For questions about this data warehouse, please contact the data engineering team.
Documentation Version: 1.0
Last Updated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

# Save documentation to file
doc_path = os.path.join(OUTPUT_DIR, 'DATA_WAREHOUSE_DOCUMENTATION.md')
with open(doc_path, 'w', encoding='utf-8') as f:
    f.write(documentation_content)
logger.info(f"✓ Data dictionary and documentation saved to: {doc_path}")

# Save data dictionary as CSV
data_dict_summary = pd.DataFrame([
    {'Table': k, 'Type': 'Dimension', 'Description': v['description'], 'Row Count': v['row_count']}
    for k, v in dimensions_doc.items()
] + [
    {'Table': k, 'Type': 'Fact', 'Description': v['description'], 'Row Count': v['row_count']}
    for k, v in facts_doc.items()
])

dict_path = os.path.join(OUTPUT_DIR, 'data_dictionary_summary.csv')
data_dict_summary.to_csv(dict_path, index=False)
logger.info(f"✓ Data dictionary summary saved to: {dict_path}")

logger.info("\n" + "=" * 80)
logger.info("✓ ALL 5 STEPS COMPLETED SUCCESSFULLY!")
logger.info("=" * 80)
logger.info("\nData Warehouse is now ready for analysis and reporting!")

2026-04-04 15:46:03,840 - INFO - 
2026-04-04 15:46:03,842 - INFO - STEP 5: DOCUMENTATION - Creating data dictionary and business glossary
2026-04-04 15:46:03,845 - INFO - ================================================================================
2026-04-04 15:46:03,847 - INFO - 
Generating data dictionary for dimensions...
2026-04-04 15:46:03,870 - INFO - ✓ Data dictionary and documentation saved to: ./dw_output\DATA_WAREHOUSE_DOCUMENTATION.md
2026-04-04 15:46:03,878 - INFO - ✓ Data dictionary summary saved to: ./dw_output\data_dictionary_summary.csv
2026-04-04 15:46:03,878 - INFO - 
2026-04-04 15:46:03,881 - INFO - ✓ ALL 5 STEPS COMPLETED SUCCESSFULLY!
2026-04-04 15:46:03,884 - INFO - ================================================================================
2026-04-04 15:46:03,886 - INFO - 
Data Warehouse is now ready for analysis and reporting!


In [58]:
# DEBUG: Check actual column names
print("=" * 80)
print("DEBUG: Checking DataFrame columns")
print("=" * 80)
print("\nfact_consumption columns:")
print(fact_consumption.columns.tolist())
print(f"Shape: {fact_consumption.shape}")
print("\nfact_consumption head:")
print(fact_consumption.head())

print("\n\ndim_location columns:")
print(dim_location.columns.tolist())
print(f"Shape: {dim_location.shape}")
print("\ndim_location head:")
print(dim_location.head())

print("\n\nfact_charging columns:")
print(fact_charging.columns.tolist())
print(f"Shape: {fact_charging.shape}")
print("\nfact_charging head:")
print(fact_charging.head())


DEBUG: Checking DataFrame columns

fact_consumption columns:
['location_id', 'hour_id', 'day_id', 'weather_id', 'energy_consumed']
Shape: (1027232, 5)

fact_consumption head:
   location_id  hour_id  day_id  weather_id  energy_consumed
0           11    41620   20362        1370              0.0
1           11    41711   20365        1373              0.0
2           11    42239   20387        1395              0.0
3           11    42123   20383        1391              0.0
4           11    41843   20371        1379              0.0


dim_location columns:
['location_id', 'municipality_code', 'municipality_name', 'parish_code', 'parish_name', 'district_code', 'district_name']
Shape: (18, 7)

dim_location head:
   location_id municipality_code            municipality_name parish_code  \
0            1               11A  Área Metropolitana do Porto        None   
1            2           11A0104                       Arouca        None   
2            3           11A0107               